# Quickstart: working with the data

This notebook loads both datasets, shows how timezones are handled, combines symbols, resamples, and computes a first statistic.
Run it from the `examples/` folder (or anywhere, after setting `REPO` below). Full reference: [`DATA.md`](../DATA.md).

**Key point:** every loader returns `ts` as a timezone-aware New York timestamp. The raw 15-year ES and gold files are in UTC,
but you never see that if you use the loaders.

In [1]:
import sys
from pathlib import Path
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()   # or set it explicitly: Path(r"C:\path\to\repo")
sys.path.insert(0, str(REPO))

from research.scripts.load import load                                    # NQ 2020-2026 research dataset
from research.scripts.build_15y import load_15y, resample                 # 15-year NQ / ES / GC
pd.set_option("display.width", 160)

## 1. NQ research dataset (2020-2026, with order flow)

`load('dev')` is the development period. `load('oot')` is the out-of-time period: keep it for final checks only (see DATA.md §7).

In [2]:
nq = load("dev")
print(nq.ts.dtype)                     # tz-aware New York
nq[["ts", "open", "high", "low", "close", "volume", "delta", "contract", "sess"]].head()

datetime64[us, America/New_York]


,ts,open,high,low,close,volume,delta,contract,sess
0,2020-01-01 18:00:00-05:00,8776.75,8779.25,8770.00,8773.5,555,84,NQH0,2020-01-02
1,2020-01-01 18:01:00-05:00,8773.25,8774.00,8770.50,8770.5,176,-2,NQH0,2020-01-02
2,2020-01-01 18:02:00-05:00,8770.25,8771.50,8769.50,8771.0,251,-45,NQH0,2020-01-02
3,2020-01-01 18:03:00-05:00,8771.50,8773.25,8771.50,8773.0,125,29,NQH0,2020-01-02
4,2020-01-01 18:04:00-05:00,8773.00,8773.25,8771.25,8771.5,73,-11,NQH0,2020-01-02


The first bar is **Wednesday 2020-01-01 18:00** but its `sess` (trading day) is **2020-01-02**: the futures day starts at 18:00 New York time.
Group by `sess`, not by calendar date.

## 2. 15-year dataset (NQ, ES, gold)

`load_15y()` builds `research/data/futures_1min_15y.parquet` on first use (about 30 seconds), then just reads it.

In [3]:
d = load_15y()
d.groupby("symbol", observed=True).agg(first=("ts", "min"), last=("ts", "max"), bars=("ts", "size"))

,first,last,bars
symbol,,,
es,2011-12-04 18:00:00-05:00,2026-07-31 16:59:00-04:00,5127103
gc,2013-07-29 09:32:00-04:00,2026-07-31 16:59:00-04:00,4280231
nq,2010-06-06 20:00:00-04:00,2026-11-24 16:59:00-05:00,5575781


### What the UTC conversion does

The raw ES file stores times in UTC. Here is the same bar in the raw file and in the loaded data:
14:30 UTC on 2 January 2024 is the **09:30 New York open**.

In [4]:
raw = pd.read_parquet(REPO / "research/data/raw_15y/es_1min_2024.parquet")
print("raw file (UTC):     ", raw[(raw.Date == "2024-01-02") & (raw.Time == "14:30:00")].iloc[0].to_dict())
es = d[d.symbol == "es"]
print("loaded (New York):  ", es[es.ts == pd.Timestamp("2024-01-02 09:30", tz="America/New_York")].iloc[0][["ts", "open", "close"]].to_dict())

raw file (UTC):      {'Date': '2024-01-02', 'Time': '14:30:00', 'Open': 4783.5, 'High': 4788.75, 'Low': 4782.25, 'Close': 4787.25, 'Volume': 20909}


loaded (New York):   {'ts': Timestamp('2024-01-02 09:30:00-0500', tz='America/New_York'), 'open': 4783.5, 'close': 4787.25}


As a sanity check, the busiest minutes of the day for each symbol should be at its real New York opens and closes
(ES/NQ 09:30 and 16:00, gold 08:30 and its 13:30 COMEX close):

In [5]:
recent = d[d.ts >= pd.Timestamp("2024-01-01", tz="America/New_York")]
minute = recent.ts.dt.strftime("%H:%M")
for sym, g in recent.groupby("symbol", observed=True):
    top = g.groupby(minute[g.index]).volume.mean().nlargest(3).index.tolist()
    print(sym, "busiest minutes (New York):", top)

es busiest minutes (New York): ['15:59', '16:00', '09:30']


gc busiest minutes (New York): ['13:29', '10:00', '08:30']
nq busiest minutes (New York): ['15:59', '09:30', '09:31']


## 3. Regular trading hours and daily bars

In [6]:
nq15 = load_15y("nq")
t = nq15.ts.dt.hour * 60 + nq15.ts.dt.minute
rth = nq15[(t >= 9 * 60 + 30) & (t < 16 * 60)]            # bars stamped 09:30 .. 15:59 (stamps are bar OPEN times)

daily_rth = rth.groupby("sess").agg(open=("open", "first"), high=("high", "max"),
                                    low=("low", "min"), close=("close", "last"), bars=("open", "size"))
daily_rth = daily_rth[daily_rth.bars >= 380]                # drop half days
daily_rth.tail()

,open,high,low,close,bars
sess,,,,,
2026-11-18,27252.25,27325.50,26921.75,27190.75,390
2026-11-19,27663.75,27838.00,26559.25,26588.75,390
2026-11-20,26322.00,26504.25,24976.50,25144.25,390
2026-11-23,26878.50,27137.00,26450.25,26818.00,390
2026-11-24,26361.25,26509.75,25998.00,26284.25,390


## 4. Combining symbols: NQ and ES side by side

In [7]:
both = load_15y(["nq", "es"])
wide = both.pivot_table(index="ts", columns="symbol", values="close", observed=True).dropna()
rets = wide.pct_change().dropna()
print(f"{len(wide):,} minutes where both traded")
print("correlation of 1-minute returns, by year:")
print(rets.groupby(rets.index.year).apply(lambda r: r.nq.corr(r.es)).round(3).to_string())

4,967,598 minutes where both traded
correlation of 1-minute returns, by year:


ts
2011    0.800
2012    0.776
2013    0.769
2014    0.822
2015    0.877
2016    0.878
2017    0.720
2018    0.895
2019    0.902
2020    0.906
2021    0.852
2022    0.940
2023    0.888
2024    0.877
2025    0.945
2026    0.921


## 5. Resampling to other bar sizes

Bars are labelled by their open time and never straddle the 18:00 session start.

In [8]:
week = nq15[nq15.sess >= "2026-11-16"]
m15 = resample(week, "15min")
h1 = resample(week, "1h")
print(len(week), "1-minute bars ->", len(m15), "15-minute bars ->", len(h1), "hourly bars")
h1.head()

9660 1-minute bars -> 644 15-minute bars -> 161 hourly bars


,symbol,ts,open,high,low,close,volume,sess
0,nq,2026-11-15 18:00:00-05:00,28062.50,28279.25,28059.25,28253.75,16382,2026-11-16
1,nq,2026-11-15 19:00:00-05:00,28253.50,28267.50,28110.50,28177.00,12401,2026-11-16
2,nq,2026-11-15 20:00:00-05:00,28178.25,28178.25,27854.25,27865.75,24772,2026-11-16
3,nq,2026-11-15 21:00:00-05:00,27864.50,28059.75,27854.75,27995.25,17435,2026-11-16
4,nq,2026-11-15 22:00:00-05:00,27994.25,28009.75,27836.75,27878.25,11374,2026-11-16


## 6. A first statistic: how often does the NQ RTH session close above its open?

Computed by year on the 15-year data. Up-day rates near 50-55% are typical; whether any rule beats that is what the research in this repo tests.

In [9]:
up = (daily_rth.close > daily_rth.open)
by_year = up.groupby(daily_rth.index.year).agg(["mean", "size"])
by_year["mean"] = (100 * by_year["mean"]).round(1)
by_year.columns = ["% up days", "sessions"]
by_year.T

sess,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
% up days,51.0,52.6,54.1,53.8,56.4,52.2,53.4,58.5,50.0,55.8,57.6,56.6,48.4,59.7,51.8,54.5,48.2
sessions,143.0,251.0,246.0,249.0,243.0,249.0,251.0,248.0,248.0,249.0,245.0,251.0,250.0,248.0,249.0,246.0,226.0


## 7. Watch out for contract rolls

Prices are raw front-month, so the series jumps when it rolls to the next contract (NQ/ES quarterly). That's fine within a session,
but distorts anything comparing prices across days. For NQ 2020-2026, `load(..., adjusted=True)` gives a back-adjusted series:

In [10]:
raw_d = load("all").groupby("sess").close.last()
adj_d = load("all", adjusted=True).groupby("sess").close.last()
rolls = pd.read_csv(REPO / "research/data/roll_table.csv", parse_dates=["roll_sess"])
r = rolls.roll_sess.iloc[-1]
i = raw_d.index.get_loc(r)
gap = rolls.gap.iloc[-1]
print(f"roll to {rolls.contract.iloc[-1]} on session {r.date()}: new contract priced {gap:+.2f} pts vs the old one")
print(f"raw close change on the roll day:      {raw_d.iloc[i] - raw_d.iloc[i-1]:+.2f}  (includes the {gap:+.2f} contract switch)")
print(f"adjusted close change on the roll day: {adj_d.iloc[i] - adj_d.iloc[i-1]:+.2f}  (the real move)")
pd.DataFrame({"raw close": raw_d.iloc[i-2:i+2], "back-adjusted close": adj_d.iloc[i-2:i+2]})

roll to NQZ6 on session 2026-09-14: new contract priced +299.25 pts vs the old one
raw close change on the roll day:      +98.00  (includes the +299.25 contract switch)
adjusted close change on the roll day: -201.25  (the real move)


,raw close,back-adjusted close
sess,,
2026-09-10,29143.25,29442.50
2026-09-11,29391.75,29691.00
2026-09-14,29489.75,29489.75
2026-09-15,29277.00,29277.00


Before the roll, the adjusted series is shifted by the gap so it joins the new contract smoothly; from the roll on, both are identical. See DATA.md §6.

**Next:** read [`DATA.md`](../DATA.md) for every file and option, and `research/output/` for the results of the studies.